# Task 1: Foundational Setup & Exploratory Data Analysis (EDA)
**Dataset:** Superstore Sales  
**Program:** ApexPlanet Software Pvt. Ltd. – Data Analytics

**Objective:** Set up the environment, clean the data and perform exploratory analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

sns.set_theme(style='whitegrid')
FIG = Path('../reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/Sample - Superstore.csv', encoding='latin-1')
print(df.shape)
df.head()

## 2. Data Cleaning
### 2.1 Missing values

In [ ]:
print(df.isna().sum())
num_cols = df.select_dtypes('number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for c in df.select_dtypes('object').columns:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode()[0])
print('Remaining missing:', df.isna().sum().sum())

### 2.2 Duplicates

In [ ]:
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

### 2.3 Fix data types

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
for c in ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']:
    df[c] = df[c].astype('category')
df.dtypes

### 2.4 Outliers (IQR method)

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - 1.5*i, q3 + 1.5*i

for c in ['Sales', 'Profit']:
    lo, hi = iqr_bounds(df[c])
    n = ((df[c] < lo) | (df[c] > hi)).sum()
    print(f'{c}: {n} outliers outside [{lo:.2f}, {hi:.2f}]')
    df[c + '_capped'] = df[c].clip(lo, hi)

### 2.5 Save cleaned dataset

In [ ]:
df.to_csv('../data/superstore_cleaned.csv', index=False)
print('Saved cleaned dataset:', df.shape)

## 3. Exploratory Data Analysis
### 3.1 Statistical summary

In [ ]:
df.info()
df.describe()

### 3.2 Histograms & boxplots (distributions)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for i, c in enumerate(['Sales', 'Profit']):
    sns.histplot(df[c + '_capped'], bins=40, kde=True, ax=ax[0, i]); ax[0, i].set_title(f'{c} distribution (IQR-capped)')
    sns.boxplot(x=df[c + '_capped'], ax=ax[1, i]); ax[1, i].set_title(f'{c} boxplot (IQR-capped)')
plt.tight_layout(); plt.savefig(FIG/'distributions.png', dpi=150); plt.show()

### 3.3 Bar charts (categorical comparisons)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
df.groupby('Category', observed=True)['Sales'].sum().sort_values().plot.barh(ax=ax[0], color='steelblue'); ax[0].set_title('Total Sales by Category')
df.groupby('Region', observed=True)['Profit'].sum().sort_values().plot.barh(ax=ax[1], color='seagreen'); ax[1].set_title('Total Profit by Region')
plt.tight_layout(); plt.savefig(FIG/'bar_charts.png', dpi=150); plt.show()

### 3.4 Heatmap (correlation matrix)

In [ ]:
corr = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix'); plt.savefig(FIG/'heatmap.png', dpi=150); plt.show()

### 3.5 Line chart (trend over time)

In [ ]:
monthly = df.set_index('Order Date')['Sales'].resample('MS').sum()
monthly.plot(marker='o', figsize=(11, 4)); plt.title('Monthly Sales Trend'); plt.ylabel('Sales')
plt.savefig(FIG/'monthly_trend.png', dpi=150); plt.show()
px.line(monthly.reset_index(), x='Order Date', y='Sales', title='Monthly Sales (interactive)').show()

## 4. Key Insights

In [ ]:
cat_sales = df.groupby('Category', observed=True)['Sales'].sum()
reg_profit = df.groupby('Region', observed=True)['Profit'].sum()
sub_profit = df.groupby('Sub-Category', observed=True)['Profit'].sum()
seg = df.groupby('Segment', observed=True)['Sales'].sum()
disc_corr = df['Discount'].corr(df['Profit'])
best_month = monthly.idxmax()

print(f'1. Top category by sales: {cat_sales.idxmax()} ({cat_sales.max():,.0f})')
print(f'2. Most profitable region: {reg_profit.idxmax()}; least: {reg_profit.idxmin()}')
print(f'3. Discount vs Profit correlation: {disc_corr:.2f}')
print(f'4. Peak sales month: {best_month:%B %Y}')
print(f'5. Least profitable sub-category: {sub_profit.idxmin()} ({sub_profit.min():,.0f}); top segment: {seg.idxmax()}')

> After running on the real data, write your own interpretation of these 5 insights here.